# FantasAI - Stage 4: Vector Search & Player Recommendations

## Overview
This notebook creates vector embeddings from player performance features and builds a similarity search system to find comparable players based on their statistical profiles.

## Objectives
1. **Feature Engineering for Embeddings**: Select and normalize ML features
2. **Dimensionality Reduction**: Create compact vector representations
3. **Vector Search Index**: Build searchable index with Databricks Vector Search
4. **Similarity Queries**: Find similar players by performance patterns
5. **Recommendations**: Identify breakout candidates and waiver wire targets

## Data Source
- **Table**: `main.fantasai.ml_player_features`
- **Records**: 29,127 player-weeks (QB/RB/WR/TE)
- **Features**: 54 engineered features (rolling stats, momentum, position-specific, opponent strength)

## Use Cases
- Find similar players to top performers
- Identify breakout candidates with similar trajectories
- Discover waiver wire gems with comparable patterns
- Position-specific similarity search

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from databricks.sdk import WorkspaceClient
import json

# Configuration
catalog = "main"
schema = "fantasai"
source_table = f"{catalog}.{schema}.ml_player_features"
embeddings_table = f"{catalog}.{schema}.player_embeddings"
vector_search_endpoint = "fantasai-vs-endpoint"

# Load ML features
print("Loading ML features...")
ml_features = spark.table(source_table)
print(f"✓ Loaded {ml_features.count():,} player-week records")
print(f"✓ Features: {len(ml_features.columns)} columns")

# Show schema overview
print("\nPosition distribution:")
display(ml_features.groupBy("position").count().orderBy("position"))

In [0]:
# Select numerical features for embeddings (exclude IDs, names, categorical text)
# Based on actual ml_player_features schema
embedding_features = [
    # Rolling statistics
    "rolling_3g_avg", "rolling_5g_avg", "rolling_10g_avg", "rolling_5g_stddev", "form_variance_ratio",
    
    # Momentum & trends
    "momentum_score", "wow_change", "scoring_streak", "season_avg_to_date",
    
    # Temporal features
    "is_early_season", "is_mid_season", "is_late_season", "is_playoff_weeks",
    "weeks_into_season", "games_played_streak", "weeks_since_last_game", "coming_off_bye",
    
    # Position-specific (QB)
    "qb_passing_yards", "qb_passing_tds", "qb_attempts", "qb_completions",
    
    # Position-specific (RB)
    "rb_carries", "rb_rushing_yards", "rb_rushing_tds",
    
    # Position-specific (receiving stats - WR/TE/RB)
    "rec_targets", "rec_receptions", "rec_yards", "rec_tds",
    
    # Rolling position metrics
    "rolling_3g_targets", "rolling_3g_carries", "rolling_3g_attempts",
    
    # Team context
    "team_offensive_strength", "team_offense_rank", "position_share_pct",
    
    # Opponent strength
    "def_points_allowed_avg", "def_rank_vs_position",
    
    # Historical matchup
    "career_avg_vs_opponent", "games_vs_opponent", "max_points_vs_opponent", "recent_avg_vs_opponent",
    
    # Season rankings
    "season_position_rank", "season_percentile"
]

print(f"Selected {len(embedding_features)} numerical features for embeddings\n")

# Fill nulls with 0 (position-specific features will be null for other positions)
print("Handling missing values...")
feature_data = ml_features.fillna(0.0, subset=embedding_features)

# Add identifier columns for later retrieval
feature_data = feature_data.select(
    "master_player_id",
    "player_name",
    "position",
    "season",
    "week",
    "team",
    "current_week_points",
    *embedding_features
)

print(f"✓ Prepared {feature_data.count():,} records for embedding generation")
display(feature_data.limit(3))

In [0]:
# Create embeddings using Spark SQL functions (compatible with Serverless)
print("Computing feature statistics for normalization...")

# Calculate mean and stddev for each feature
stats_exprs = []
for feat in embedding_features:
    stats_exprs.append(F.avg(feat).alias(f"{feat}_mean"))
    stats_exprs.append(F.stddev(feat).alias(f"{feat}_std"))

stats = feature_data.agg(*stats_exprs).first()

print("✓ Computed statistics for normalization\n")

# Normalize features: (x - mean) / std
print("Normalizing features...")
normalized_cols = []
for feat in embedding_features:
    mean_val = stats[f"{feat}_mean"] or 0.0
    std_val = stats[f"{feat}_std"] or 1.0
    
    # Avoid division by zero
    if std_val == 0 or std_val is None:
        std_val = 1.0
    
    normalized_cols.append(
        ((F.col(feat) - F.lit(mean_val)) / F.lit(std_val)).alias(f"{feat}_norm")
    )

feature_data_norm = feature_data.select(
    "master_player_id",
    "player_name",
    "position",
    "season",
    "week",
    "team",
    "current_week_points",
    *normalized_cols
)

print("✓ Normalized all features\n")

# Create embedding array from normalized features
print("Creating embedding vectors...")
norm_feature_names = [f"{feat}_norm" for feat in embedding_features]

embeddings_df = feature_data_norm.withColumn(
    "embedding_array",
    F.array(*[F.col(feat) for feat in norm_feature_names])
).drop(*norm_feature_names)  # Drop individual normalized columns

# Create composite primary key (player + season + week)
embeddings_df = embeddings_df.withColumn(
    "id",
    F.concat_ws("_", F.col("master_player_id"), F.col("season"), F.col("week"))
)

print(f"✓ Generated {len(embedding_features)}-dimensional embeddings")
print(f"✓ Total records: {embeddings_df.count():,}")

# Show sample
print("\nSample embeddings (showing first 2 dimensions):")
display(
    embeddings_df.select(
        "id", "player_name", "position", "season", "week", 
        "current_week_points",
        F.slice(F.col("embedding_array"), 1, 2).alias("embedding_preview")
    ).limit(5)
)

In [0]:
# Prepare final embeddings table with all required columns
# Ensure id column is not nullable for primary key constraint
final_embeddings = embeddings_df.select(
    F.col("id").cast("string").alias("id"),  # Primary key - ensure NOT NULL
    "master_player_id",
    "player_name",
    "position",
    "season",
    "week",
    "team",
    "current_week_points",
    "embedding_array"
)

print(f"Writing embeddings to {embeddings_table}...")

# Write to Delta table
final_embeddings.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(embeddings_table)

print(f"✓ Wrote {spark.table(embeddings_table).count():,} records to {embeddings_table}")

# Alter table to set id column as NOT NULL first
print("\nSetting id column as NOT NULL...")
spark.sql(f"""
    ALTER TABLE {embeddings_table}
    ALTER COLUMN id SET NOT NULL
""")

# Enable Change Data Feed (required for Delta Sync)
print("Enabling Change Data Feed...")
spark.sql(f"""
    ALTER TABLE {embeddings_table}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Add primary key constraint (required for Vector Search)
print("Adding primary key constraint...")
spark.sql(f"""
    ALTER TABLE {embeddings_table}
    ADD CONSTRAINT player_embeddings_pk PRIMARY KEY (id)
""")

print("\n✓ Table configured for Vector Search (CDF enabled, primary key set)")

# Verify table
print("\nTable summary:")
display(spark.sql(f"""
    SELECT 
        position,
        COUNT(*) as total_embeddings,
        COUNT(DISTINCT master_player_id) as unique_players,
        AVG(current_week_points) as avg_points
    FROM {embeddings_table}
    GROUP BY position
    ORDER BY position
"""))

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

w = WorkspaceClient()

# Define FantasAI endpoint name
fantasai_endpoint_name = "fantasai-vs-endpoint"

# List existing endpoints
print("Checking existing Vector Search endpoints...")
endpoints_list = list(w.vector_search_endpoints.list_endpoints())

# Delete any existing endpoints to make room for FantasAI endpoint
for ep in endpoints_list:
    print(f"\nDeleting existing endpoint: {ep.name}")
    w.vector_search_endpoints.delete_endpoint(ep.name)
    print(f"✓ Deleted {ep.name}")
    time.sleep(5)  # Brief wait for cleanup

# Create new FantasAI endpoint
print("\nCreating FantasAI Vector Search endpoint...")
endpoint = w.vector_search_endpoints.create_endpoint(
    name=fantasai_endpoint_name,
    endpoint_type=EndpointType.STANDARD
)

print(f"✓ Endpoint creation initiated: {fantasai_endpoint_name}")

# Wait for endpoint to be online
print("\nWaiting for endpoint to be online...")
max_wait = 600  # 10 minutes
elapsed = 0
while elapsed < max_wait:
    endpoint = w.vector_search_endpoints.get_endpoint(fantasai_endpoint_name)
    status = str(endpoint.endpoint_status.state)
    print(f"  Status: {status} (elapsed: {elapsed}s)")
    
    if "ONLINE" in status:
        print("✓ Endpoint is online and ready")
        break
    elif any(s in status for s in ["PROVISIONING", "OFFLINE"]):
        time.sleep(30)
        elapsed += 30
    else:
        print(f"⚠ Unexpected state: {status}")
        time.sleep(30)
        elapsed += 30

if elapsed >= max_wait:
    print("⚠ Endpoint creation timed out")
else:
    print(f"\nFantasAI Vector Search Endpoint Ready:")
    print(f"  Name: {endpoint.name}")
    print(f"  Type: {endpoint.endpoint_type}")
    print(f"  Status: {endpoint.endpoint_status.state}")

# Update global variable for use in subsequent cells
vector_search_endpoint = fantasai_endpoint_name

In [0]:
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingVectorColumn,
    PipelineType,
    VectorIndexType
)
import time

# Configuration (ensure variables are defined)
catalog = "main"
schema = "fantasai"
embeddings_table = f"{catalog}.{schema}.player_embeddings"
index_name = f"{catalog}.{schema}.fantasai_player_index"  # NEW NAME for fresh start

# Embedding features count (from previous cell)
embedding_features_count = 42

print(f"Creating FantasAI Vector Search index: {index_name}")

# Check if index already exists
index_exists = False
try:
    existing_index = w.vector_search_indexes.get_index(index_name)
    print(f"✓ Index '{index_name}' already exists")
    print(f"  Status: {existing_index.status.detailed_state}")
    index_exists = True
except Exception:
    print(f"Index does not exist yet, creating new one...")

if not index_exists:
    print(f"\nCreating index '{index_name}' on {vector_search_endpoint}...")
    
    # Create Delta Sync spec with proper SDK types
    delta_sync_spec = DeltaSyncVectorIndexSpecRequest(
        source_table=embeddings_table,
        embedding_vector_columns=[
            EmbeddingVectorColumn(
                name="embedding_array",
                embedding_dimension=embedding_features_count
            )
        ],
        pipeline_type=PipelineType.TRIGGERED
    )
    
    # Create Delta Sync index with self-managed embeddings
    index = w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=vector_search_endpoint,
        primary_key="id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=delta_sync_spec
    )
    
    print("✓ Index creation initiated")
    
    # Wait for index to be ready
    print("\nWaiting for index to be online (this may take several minutes)...")
    max_wait = 900  # 15 minutes
    elapsed = 0
    while elapsed < max_wait:
        try:
            index_status = w.vector_search_indexes.get_index(index_name)
            state = str(index_status.status.detailed_state)
            print(f"  Status: {state} (elapsed: {elapsed}s)")
            
            if "ONLINE" in state:
                print("✓ Index is online and ready")
                break
            elif any(s in state for s in ["PROVISIONING", "SYNCING", "FORMATTING"]):
                time.sleep(30)
                elapsed += 30
            else:
                print(f"  Index in state: {state}, continuing to wait...")
                time.sleep(30)
                elapsed += 30
        except Exception as e:
            print(f"  Waiting for index to be available... ({elapsed}s)")
            time.sleep(30)
            elapsed += 30
    
    if elapsed >= max_wait:
        print("⚠ Index creation timed out, but may still be provisioning in background")
        print("  Check status with: w.vector_search_indexes.get_index(index_name)")
    else:
        print(f"\n✓ FantasAI Vector Search index ready!")
else:
    print(f"\nIndex already configured and ready to use.")

In [0]:
def find_similar_players(
    player_name,
    season=2024,
    week=None,
    top_k=10,
    position_filter=None,
    min_points=None
):
    """
    Find players with similar performance patterns.
    
    Args:
        player_name: Name of the reference player
        season: Season year (default: 2024)
        week: Specific week (optional, uses latest if None)
        top_k: Number of similar players to return
        position_filter: Filter by position (e.g., "WR", "RB")
        min_points: Minimum points threshold for candidates
    
    Returns:
        DataFrame with similar players and similarity scores
    """
    from databricks.vector_search.client import VectorSearchClient
    
    # FantasAI configuration
    embeddings_table = "main.fantasai.player_embeddings"
    index_name = "main.fantasai.fantasai_player_index"
    vector_search_endpoint = "fantasai-vs-endpoint"
    
    # Get the reference player's embedding
    query = f"""
        SELECT id, embedding_array, player_name, position, season, week, current_week_points
        FROM {embeddings_table}
        WHERE LOWER(player_name) LIKE LOWER('%{player_name}%')
        AND season = {season}
    """
    
    if week:
        query += f" AND week = {week}"
    else:
        query += f" ORDER BY week DESC LIMIT 1"
    
    reference = spark.sql(query).first()
    
    if not reference:
        print(f"Player '{player_name}' not found in season {season}")
        return None
    
    print(f"Reference: {reference.player_name} ({reference.position}) - Week {reference.week}, {reference.current_week_points:.1f} pts")
    print(f"Searching for top {top_k} similar players...\n")
    
    # Query vector search index
    vsc = VectorSearchClient()
    index = vsc.get_index(endpoint_name=vector_search_endpoint, index_name=index_name)
    
    # Build filters
    filters = None
    if position_filter:
        filters = f"position = '{position_filter}'"
    if min_points:
        filter_clause = f"current_week_points >= {min_points}"
        filters = f"{filters} AND {filter_clause}" if filters else filter_clause
    
    # Similarity search
    results = index.similarity_search(
        query_vector=reference.embedding_array,
        columns=["id", "player_name", "position", "season", "week", "team", "current_week_points"],
        num_results=top_k + 1,  # +1 to exclude the reference player
        filters=filters
    )
    
    # Convert to DataFrame
    results_data = results.get('result', {}).get('data_array', [])
    if not results_data:
        print("No similar players found")
        return None
    
    # Parse results (last column is similarity score)
    similar_players = []
    for row in results_data:
        if row[1] != reference.player_name:  # Exclude the reference player
            similar_players.append({
                "player_name": row[1],
                "position": row[2],
                "season": row[3],
                "week": row[4],
                "team": row[5],
                "points": row[6],
                "similarity_score": row[-1]
            })
    
    # Create DataFrame
    from pyspark.sql import Row
    similar_df = spark.createDataFrame([Row(**p) for p in similar_players[:top_k]])
    
    return similar_df

print("✓ Similarity search function ready")
print("\nUsage: find_similar_players('Josh Allen', season=2024, top_k=10, position_filter='QB')")
print("\nNote: Index is still provisioning. Wait for ONLINE status before running similarity searches.")

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import Row
from pyspark.sql import functions as F

print("=" * 70)
print("EXAMPLE 1: Find QBs similar to Josh Allen (2024)")
print("=" * 70)
print()

# Initialize Workspace Client
w = WorkspaceClient()

# FantasAI configuration
embeddings_table = "main.fantasai.player_embeddings"
index_name = "main.fantasai.fantasai_player_index"
vector_search_endpoint = "fantasai-vs-endpoint"

# Get the reference player's embedding
reference_query = f"""
    SELECT id, embedding_array, player_name, position, season, week, current_week_points
    FROM {embeddings_table}
    WHERE LOWER(player_name) LIKE LOWER('%Josh Allen%')
    AND season = 2024
    AND week = 14
    LIMIT 1
"""

reference = spark.sql(reference_query).first()

if not reference:
    print("Player not found")
else:
    print(f"Reference: {reference.player_name} ({reference.position}) - Week {reference.week}, {reference.current_week_points:.1f} pts")
    print(f"Searching for top 10 similar QBs...\n")
    
    # Query vector search index using SDK
    results = w.vector_search_indexes.query_index(
        index_name=index_name,
        query_vector=reference.embedding_array,
        columns=["id", "player_name", "position", "season", "week", "team", "current_week_points"],
        num_results=11,  # +1 to exclude the reference player
        filters_json='{"position": "QB"}'
    )
    
    # Parse results
    if results.result and results.result.data_array:
        similar_players = []
        for row in results.result.data_array:
            if row[1] != reference.player_name:  # Exclude the reference player
                similar_players.append({
                    "player_name": row[1],
                    "position": row[2],
                    "season": row[3],
                    "week": row[4],
                    "team": row[5],
                    "points": row[6],
                    "similarity_score": row[-1]
                })
        
        # Create DataFrame
        similar_df = spark.createDataFrame([Row(**p) for p in similar_players[:10]])
        
        print("\nTop 10 similar quarterbacks:")
        display(similar_df.orderBy(F.col("similarity_score").desc()))
    else:
        print("No similar players found")

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import Row
from pyspark.sql import functions as F

print("=" * 70)
print("EXAMPLE 2: Find RBs similar to Christian McCaffrey (2024)")
print("=" * 70)
print()

# Initialize Workspace Client
w = WorkspaceClient()

# FantasAI configuration
embeddings_table = "main.fantasai.player_embeddings"
index_name = "main.fantasai.fantasai_player_index"
vector_search_endpoint = "fantasai-vs-endpoint"

# Get the reference player's embedding
reference_query = f"""
    SELECT id, embedding_array, player_name, position, season, week, current_week_points
    FROM {embeddings_table}
    WHERE LOWER(player_name) LIKE LOWER('%McCaffrey%')
    AND season = 2024
    ORDER BY week DESC
    LIMIT 1
"""

reference = spark.sql(reference_query).first()

if not reference:
    print("Player not found")
else:
    print(f"Reference: {reference.player_name} ({reference.position}) - Week {reference.week}, {reference.current_week_points:.1f} pts")
    print(f"Searching for top 10 similar RBs...\n")
    
    # Query vector search index using SDK
    results = w.vector_search_indexes.query_index(
        index_name=index_name,
        query_vector=reference.embedding_array,
        columns=["id", "player_name", "position", "season", "week", "team", "current_week_points"],
        num_results=11,  # +1 to exclude the reference player
        filters_json='{"position": "RB"}'
    )
    
    # Parse results
    if results.result and results.result.data_array:
        similar_players = []
        for row in results.result.data_array:
            if row[1] != reference.player_name:  # Exclude the reference player
                similar_players.append({
                    "player_name": row[1],
                    "position": row[2],
                    "season": row[3],
                    "week": row[4],
                    "team": row[5],
                    "points": row[6],
                    "similarity_score": row[-1]
                })
        
        # Create DataFrame
        similar_df = spark.createDataFrame([Row(**p) for p in similar_players[:10]])
        
        print("\nTop 10 similar running backs:")
        display(similar_df.orderBy(F.col("similarity_score").desc()))
    else:
        print("No similar players found")

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql import Row
from pyspark.sql import functions as F

print("=" * 70)
print("EXAMPLE 3: Find breakout WR candidates similar to Puka Nacua")
print("=" * 70)
print()

# Initialize Workspace Client
w = WorkspaceClient()

# FantasAI configuration
embeddings_table = "main.fantasai.player_embeddings"
index_name = "main.fantasai.fantasai_player_index"
vector_search_endpoint = "fantasai-vs-endpoint"

# Get the reference player's embedding
reference_query = f"""
    SELECT id, embedding_array, player_name, position, season, week, current_week_points
    FROM {embeddings_table}
    WHERE LOWER(player_name) LIKE LOWER('%Nacua%')
    AND season = 2024
    ORDER BY week DESC
    LIMIT 1
"""

reference = spark.sql(reference_query).first()

if not reference:
    print("Player not found")
else:
    print(f"Reference: {reference.player_name} ({reference.position}) - Week {reference.week}, {reference.current_week_points:.1f} pts")
    print(f"Searching for top 15 similar WRs...\n")
    
    # Query vector search index using SDK
    results = w.vector_search_indexes.query_index(
        index_name=index_name,
        query_vector=reference.embedding_array,
        columns=["id", "player_name", "position", "season", "week", "team", "current_week_points"],
        num_results=16,  # +1 to exclude the reference player
        filters_json='{"position": "WR", "current_week_points": {"$gte": 5.0}}'
    )
    
    # Parse results
    if results.result and results.result.data_array:
        similar_players = []
        for row in results.result.data_array:
            if row[1] != reference.player_name:  # Exclude the reference player
                similar_players.append({
                    "player_name": row[1],
                    "position": row[2],
                    "season": row[3],
                    "week": row[4],
                    "team": row[5],
                    "points": row[6],
                    "similarity_score": row[-1]
                })
        
        # Create DataFrame
        similar_df = spark.createDataFrame([Row(**p) for p in similar_players[:15]])
        
        print("\nTop 15 WRs with similar breakout patterns:")
        display(similar_df.orderBy(F.col("similarity_score").desc()))
    else:
        print("No similar players found")

In [0]:
from databricks.sdk import WorkspaceClient

print("=" * 70)
print("FANTASAI VECTOR SEARCH VALIDATION")
print("=" * 70)
print()

# Initialize Workspace Client
w = WorkspaceClient()

# FantasAI configuration
embeddings_table = "main.fantasai.player_embeddings"
index_name = "main.fantasai.fantasai_player_index"
vector_search_endpoint = "fantasai-vs-endpoint"

# Check index status
index_status = w.vector_search_indexes.get_index(index_name=index_name)
print(f"Index: {index_name}")
print(f"Status: {index_status.status}")
print(f"Endpoint: {vector_search_endpoint}")
print()

# Embeddings table statistics
print("Embeddings Table Statistics:")
stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_embeddings,
        COUNT(DISTINCT master_player_id) as unique_players,
        COUNT(DISTINCT position) as positions,
        COUNT(DISTINCT season) as seasons,
        COUNT(DISTINCT week) as weeks
    FROM {embeddings_table}
""").first()

print(f"  Total Embeddings: {stats.total_embeddings:,}")
print(f"  Unique Players: {stats.unique_players:,}")
print(f"  Positions: {stats.positions}")
print(f"  Seasons: {stats.seasons}")
print(f"  Weeks: {stats.weeks}")
print()

# Position breakdown
print("Position Breakdown:")
display(spark.sql(f"""
    SELECT 
        position,
        COUNT(*) as embeddings,
        COUNT(DISTINCT master_player_id) as players,
        AVG(current_week_points) as avg_points,
        MAX(current_week_points) as max_points
    FROM {embeddings_table}
    GROUP BY position
    ORDER BY position
"""))

print("\n✓ FantasAI Vector Search system ready for player similarity queries!")